In [ ]:
import pickle
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
import liana as li
import cell2cell as c2c
from pathlib import Path
import anndata as ad
import plotnine as p9
import decoupler as dc # needed for pathway enrichment
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict

%matplotlib inline


use_gpu = True
if use_gpu:
    import torch
    import tensorly as tl
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        tl.set_backend('pytorch')
else:
    device = "cpu"

OUTDIR = Path('data')

## Tensor-cell2cell

https://liana-py.readthedocs.io/en/latest/notebooks/liana_c2c.html

Загрузка данных

In [ ]:
liana_res_filt = pd.read_parquet(OUTDIR / 'liana_res_filt.parquet')
context_df = pd.read_parquet(OUTDIR / 'context_df.parquet')

with open(OUTDIR / 'context_dict.pkl', 'rb') as f:
    context_dict = defaultdict(lambda: 'Unknown', pickle.load(f))

tensor = li.multi.to_tensor_c2c(
    liana_res=liana_res_filt,
    sample_key='donor_id',
    score_key='magnitude_rank',
    non_expressed_fill=0,
    how='outer'
)

print(tensor.shape)

context_map = dict(zip(context_df['donor_id'], context_df['PAM50_subtype']))
context_map = defaultdict(lambda: 'Unknown', context_map)

tensor_meta = c2c.tensor.generate_tensor_metadata(
    interaction_tensor=tensor,
    metadata_dicts=[context_map, None, None, None],
    fill_with_order_elements=True
)

print(tensor_meta)

In [ ]:
tensor = c2c.analysis.run_tensor_cell2cell_pipeline(tensor,
                                                    tensor_meta,
                                                    backend='pytorch',
                                                    device='cpu',
                                                    copy_tensor=True, # Whether to output a new tensor or modifying the original
                                                    rank=None, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis. Here, it was precomuputed.
                                                    tf_optimization='regular', # To define how robust we want the analysis to be. 
                                                    random_state=0, # Random seed for reproducibility
                                                    #device=device, # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                                    elbow_metric='error', # Metric to use in the elbow analysis.
                                                    smooth_elbow=False, # Whether smoothing the metric of the elbow analysis.
                                                    upper_rank=20, # Max number of factors to try in the elbow analysis
                                                    tf_init='random', # Initialization method of the tensor factorization
                                                    tf_svd='numpy_svd', # Type of SVD to use if the initialization is 'svd'
                                                    cmaps=None, # Color palettes to use in color each of the dimensions. Must be a list of palettes.
                                                    sample_col='Element', # Columns containing the elements in the tensor metadata
                                                    group_col='Category', # Columns containing the major groups in the tensor metadata
                                                    output_fig=False, # Whether to output the figures. If False, figures won't be saved a files if a folder was passed in output_folder.
                                                    )

Running Elbow Analysis


 10%|█         | 2/20 [33:05<5:04:55, 1016.44s/it]

In [ ]:
print(tensor.factors['Contexts'].shape)
print(tensor.factors['Ligand-Receptor Pairs'].shape)

In [ ]:
tensor = c2c.analysis.run_tensor_cell2cell_pipeline(tensor,
                                                    tensor_meta,
                                                    copy_tensor=True, # Whether to output a new tensor or modifying the original
                                                    rank=11, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis. Here, it was precomuputed.
                                                    tf_optimization='regular', # To define how robust we want the analysis to be. 
                                                    random_state=0, # Random seed for reproducibility
                                                    device=device, # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                                    elbow_metric='error', # Metric to use in the elbow analysis.
                                                    smooth_elbow=False, # Whether smoothing the metric of the elbow analysis.
                                                    upper_rank=20, # Max number of factors to try in the elbow analysis
                                                    tf_init='random', # Initialization method of the tensor factorization
                                                    tf_svd='numpy_svd', # Type of SVD to use if the initialization is 'svd'
                                                    cmaps=None, # Color palettes to use in color each of the dimensions. Must be a list of palettes.
                                                    sample_col='Element', # Columns containing the elements in the tensor metadata
                                                    group_col='Category', # Columns containing the major groups in the tensor metadata
                                                    output_fig=False, # Whether to output the figures. If False, figures won't be saved a files if a folder was passed in output_folder.
                                                    )

In [ ]:
factors, axes = c2c.plotting.tensor_factors_plot(interaction_tensor=tensor,
                                                 metadata = tensor_meta, # This is the metadata for each dimension
                                                 sample_col='Element',
                                                 group_col='Category',
                                                 meta_cmaps = ['viridis', 'Dark2_r', 'tab20', 'tab20'],
                                                 fontsize=10, # Font size of the figures generated
                                                 )

In [ ]:
factors = tensor.factors

In [ ]:
factors.keys()

In [ ]:
_ = c2c.plotting.context_boxplot(context_loadings=factors['Contexts'],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(12, 6),
                                 statistical_test='t-test_ind',
                                 pval_correction='fdr_bh',
                                 cmap='plasma',
                                 verbose=False,
                                )

In [ ]:
c2c.plotting.ccc_networks_plot(factors,
                               included_factors=['Factor 7'],
                               network_layout='circular',
                               ccc_threshold=0.05, # Only important communication
                               nrows=1,
                               panel_size=(8, 8), # This changes the size of each figure panel.
                              )

In [ ]:
lr_loadings = factors['Ligand-Receptor Pairs']
lr_loadings.sort_values("Factor 2", ascending=False).head(20)

## Downstream Analysis

In [ ]:
# load PROGENy pathways
net = dc.op.progeny(organism='human', top=5000)

In [ ]:
# load full list of ligand-receptor pairs
lr_pairs = li.resource.select_resource('consensus')

In [ ]:
# generate ligand-receptor geneset
lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^").rename(columns = {"interaction": "target"})
lr_progeny.head()

In [ ]:
# run enrichment analysis
estimate, pvals = dc.mt.ulm(lr_loadings.transpose(), lr_progeny, raw=False)

In [ ]:
dc.pl.barplot(estimate, 'Factor 2', vertical=False, cmap='coolwarm', vmin=-7, vmax=7)

факторы, связанные с Basal vs LumA

In [ ]:
ctx = factors['Contexts'].copy()
ctx['subtype'] = ctx.index.map(context_dict)

rows = []
for fac in factors['Contexts'].columns:
    basal = ctx.loc[ctx['subtype'] == 'Basal', fac]
    luma = ctx.loc[ctx['subtype'] == 'LumA', fac]

    _, pval = mannwhitneyu(basal, luma, alternative='two-sided')

    rows.append({
        'factor': fac,
        'Basal_mean': basal.mean(),
        'LumA_mean': luma.mean(),
        'delta_Basal_minus_LumA': basal.mean() - luma.mean(),
        'enriched_in': 'Basal' if basal.mean() > luma.mean() else 'LumA',
        'pval': pval
    })

factor_stats = pd.DataFrame(rows)
factor_stats['padj'] = multipletests(factor_stats['pval'], method='fdr_bh')[1]
factor_stats = factor_stats.sort_values(['padj', 'delta_Basal_minus_LumA'], ascending=[True, False])

factor_stats

In [ ]:
target_factor = 'Factor 9'

sender_tbl = (
    factors['Sender Cells'][target_factor]
    .rename('sender_loading')
    .rename_axis('source')
    .reset_index()
    .sort_values('sender_loading', ascending=False)
)

receiver_tbl = (
    factors['Receiver Cells'][target_factor]
    .rename('receiver_loading')
    .rename_axis('target')
    .reset_index()
    .sort_values('receiver_loading', ascending=False)
)

lr_tbl = (
    factors['Ligand-Receptor Pairs'][target_factor]
    .rename('lr_loading')
    .rename_axis('interaction')
    .reset_index()
    .sort_values('lr_loading', ascending=False)
)

# sender_tbl.head(10)
receiver_tbl.head(10)
# lr_tbl.head(30)

In [ ]:
edge_tbl = sender_tbl.merge(receiver_tbl, how='cross')
edge_tbl['edge_score'] = edge_tbl['sender_loading'] * edge_tbl['receiver_loading']
edge_tbl = edge_tbl.sort_values('edge_score', ascending=False)

edge_tbl.head(20)

In [ ]:
triplet_tbl = edge_tbl[['source', 'target', 'edge_score']].merge(lr_tbl, how='cross')
triplet_tbl['triplet_score'] = triplet_tbl['edge_score'] * triplet_tbl['lr_loading']
triplet_tbl = triplet_tbl.sort_values('triplet_score', ascending=False)

triplet_tbl.head(50)

In [ ]:
triplet_tbl.query("source == 'Malignant cell' or target == 'Malignant cell'").head(10)

In [ ]:
edge_tbl.query("source == 'Malignant cell' or target == 'Malignant cell'").head(10)

In [ ]:
from statsmodels.stats.multitest import multipletests

pathway_tbl = pd.DataFrame({
    'pathway': estimate.columns,
    'estimate': estimate.loc[target_factor].values,
    'pval': pvals.loc[target_factor].values
})

pathway_tbl['padj'] = multipletests(pathway_tbl['pval'], method='fdr_bh')[1]
pathway_tbl = pathway_tbl.sort_values(['padj', 'estimate'], ascending=[True, False])

pathway_tbl.head(20)